<a href="https://colab.research.google.com/github/dowes48/A2E_polars/blob/main/A2E_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fast A2E Using Polars - Google Colab Session

When you run one of the following Colab code cells for the first time, Google will warn you that this is not a Google notebook. Choose the option "Run Anyway".

For example, **Click on the arrow in the next cell** to determine what version of Python Google is using, then choose the option "Run Anyway".  

*one can also run a code cell by placing the cursor in the cell and keying "Shift-Enter"*

In [1]:
!python --version

Python 3.12.12


Python 3.12.12 was released in 10/2025 as a security fix for version 3.12 which was first released in 10/2023. Since the current stable version of Python is 3.14, one may occasionally find that the latest Python expressions may not work in Colab. See https://docs.python.org/release/3.12.12/ for version specific documentation.

*Colab can also be configured to run R code.*

---

I am using GitHub to store this Notebook and its associated data files. If you got this far, you already have a Colab copy of my notebook. Now you need to do next is to **click the arrow in the next cell** to make a temporary copy of my entire GitHub repository including the data files that we will want to use.

In [2]:
!git clone "https://github.com/dowes48/A2E_polars"

Cloning into 'A2E_polars'...
remote: Enumerating objects: 37, done.
remote: Counting objects: 100% (37/37), done.
remote: Compressing objects: 100% (32/32), done.
remote: Total 37 (delta 14), reused 25 (delta 5), pack-reused 0 (from 0)
Receiving objects: 100% (37/37), 4.00 MiB | 3.77 MiB/s, done.
Resolving deltas: 100% (14/14), done.


After the cloning has finished, click on the File folder icon to the left. A folder for A2E_polars should appear. Under this, you will find my data folder.

---

This is a "Text" cell - actually created as Markdown language. Colab is very nice to use for **Markdown** because while the Markdown special characters are entered in the left-hand side of this cell, the right-hand side shows how the text will appear to the reader. Double-click on this cell to enter editing mode and you will see what I mean.

---

The following following cell *imports* the libraries that will be used in this notebook. The cells that follow depend on the libraries' code, so make certain this is run first. In general, run cells in the given sequence.

In [3]:
import os                                # basic file handling functions
import polars as pl                      # polars, a DataFrame library for manipulating structured data
from datetime import timedelta           # a single datatype needed for calculations
import time

In [6]:
# read the data file into a polars dataframe ('pl' is shorthand for polars)
df = pl.read_csv(r'A2E_polars/data/HRA_data.csv', try_parse_dates = True)

In [ ]:
# rename exam year column
df = df.rename({'examY': 'year'})

# define Enumeration datatypes
enum_vs   = pl.Enum(["alive", "dead"])
enum_sex  = pl.Enum(["M", "F"])
enum_dx   = pl.Enum(["healthy", "cancer", "cardiac", "diabetes", "neuro", "pulm", "renal", "other"])
enum_site = pl.Enum(["AL", "AZ", "CA", "FL", "GA", "LA", "SC", "TX", "VA"])
enum_agebnd  = pl.Enum(["65 - 69", "70 - 74", "75 - 79", "80 - 84"])

# cast column values to defined datatypes
df = df.with_columns(
    pl.col("vs").cast(enum_vs),
    pl.col("sex").cast(enum_sex),
    pl.col("dxGrp").cast(enum_dx),
    pl.col("site").cast(enum_site),
    pl.col("ageBand").cast(enum_agebnd))

df = df.with_columns(
    pl.col("age").cast(pl.UInt8),
    pl.col("year").cast(pl.UInt16),
    pl.col("UID").cast(pl.UInt32),
    pl.col("mr").cast(pl.Float32))

print(df)

In [ ]:
# split off columns of "details"
df_details = df.select('UID', 'sex', 'mr', 'dxGrp', 'site', 'ageBand')
print("\n\ndetails dataframe 'df_details' variables will be re-joined asfter expansion")
print(df_details)

# keep columns needed for expansion
df = df.select('UID', 'examD', 'exitD', 'vs', 'age', 'year')
print("\nStudy dataframe 'df' now contains only variables needed for expansion")
print(df)

In [ ]:
# tDelta constants to be used in exposure calculation
tDelta_1yr = timedelta(days=365, hours=6)
tDelta_1day  = timedelta(days=1)

# calculate total number of intervals for each subject
df = df.with_columns(totalIntvls = ((pl.col('exitD') - pl.col('examD') + tDelta_1day)/tDelta_1yr)
                     .ceil().cast(pl.Int32))
print(df)

In [ ]:
# create temporary column of identical interval index lists
df = df.with_columns(idx_list = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11])

# take slice of idx_list and store list in new column ‘intvl’
df = df.with_columns(intvl = pl.col('idx_list').list.slice(0, pl.col('totalIntvls')))
df = df.drop('idx_list')

#  explode to get desired tabular output
df = df.explode(pl.col('intvl'))

print(df)

In [ ]:
# create temporary column of offset years, then calculate ‘intvl_date’
df = df.with_columns(offsetYrs = (pl.col("intvl") - 1).cast(pl.String))
df = df.with_columns(intvl_date = pl.col('examD').dt.offset_by((pl.col('offsetYrs')) + "y"),
        intvl_age = pl.col('age') + pl.col('intvl') - 1)
## df = df.filter(pl.col('exitD') > pl.col('intvl_date'))
print(df)


In [ ]:
# three important calculations: interval year, exposure, and actual
df = df.with_columns(
    # first the interval year (for acquiring qx)
    year = pl.col('intvl_date').dt.year(),

    # second the exposure
    persYrs = pl
        .when((pl.col('intvl_date').dt.offset_by('1y')) <= pl.col('exitD'))
        .then(pl.lit(1))
        .otherwise((pl.col('exitD') - pl.col('intvl_date') + tDelta_1day) / tDelta_1yr),

    # actual deaths
    actual = pl
        .when((pl.col('vs') == "dead") & (pl.col('intvl')
            == pl.col('totalIntvls')))
        .then(pl.lit(1))
        .otherwise(pl.lit(0)))
print(df)

In [ ]:
# clean up and get ready for join
df = df.select('UID', 'vs', 'intvl', 'intvl_date', 'intvl_age',
                         'year', 'persYrs', 'actual')
df = df.rename({'intvl_age': 'age'})
df = df.join(df_details, on='UID', how='left')
print(df)

# load mortality dataframe
df_mort = USMort.both_sexes_1933_2023()
print("\nHMD: USA, both sexes, 1933-2023")
print(df_mort)

# join to get qx
df = df.join(df_mort, on=('year','age','sex'), how='left')
print(df)


In [ ]:
# last calculation, expected deaths
df = df.with_columns(expected = pl.col('persYrs') * pl.col('qx') * pl.col('mr')).sort('UID', 'intvl')

df = df.select('UID', 'age', 'sex', 'mr', 'dxGrp', 'site', 'ageBand', 'intvl', 'intvl_date', 'year',
               'persYrs', 'actual', 'qx', 'expected').sort('UID', 'intvl')

# -----------------------------------------------------
# print("\nthe 7 leftmost columns")
print(df[:,:7])

with pl.Config(
    tbl_cell_numeric_alignment="RIGHT",
    float_precision=5,
):
    print("\nthe 7 rightmost columns")
    print(df[:,7:])
# ----------------------------------------------------

df.write_csv(r'data/a2e_results_' + '.csv')
# double-clicking the csv file will open it in MS Excel
